# Statistics Helpers — Companion Notebook

**Module:** `stats_helpers.py`  

This notebook demonstrates how to use the helper functions for:

- Z and t statistics, p-values, and critical values
- Confidence intervals and margins of error (means & proportions)
- One- and two-sample hypothesis tests (means & proportions)
- Sample size calculations
- Central Limit Theorem (CLT) simulation with visualization

> **Note:** Functions using Student's *t* distribution require **SciPy**. CLT simulation and two-sample *t* test require **NumPy**.


## Setup
Run the cell below to import the library and check which dependencies are available.
If SciPy is not available, t-based functions (like `ci_mean_unknown_sigma`, `one_sample_t_test`) will raise an ImportError.

If you need to install packages, uncomment the pip lines and run them in this notebook.

In [ ]:
# Optional installs (uncomment if running locally and needed)
# %pip install numpy scipy matplotlib

import importlib, sys

try:
    import numpy as np
    print('NumPy available:', np.__version__)
except Exception as e:
    np = None
    print('NumPy not available:', e)

try:
    import scipy
    SCIPY_AVAILABLE = True
    print('SciPy available:', scipy.__version__)
except Exception as e:
    SCIPY_AVAILABLE = False
    print('SciPy not available:', e)

try:
    import matplotlib.pyplot as plt
    print('Matplotlib available:', plt.matplotlib.__version__)
except Exception as e:
    plt = None
    print('Matplotlib not available:', e)

# Import the helpers from the local module (ensure stats_helpers.py is on sys.path)
from stats_helpers import *
print('stats_helpers imported.')


## Quick Reference
- **Z critical:** `z_critical(confidence)`
- **t critical:** `t_critical(df, confidence)` *(SciPy)*
- **Z p-value:** `p_value_z(z, alternative)`
- **t p-value:** `p_value_t(t, df, alternative)` *(SciPy)*
- **CI mean (known σ):** `ci_mean_known_sigma(xbar, sigma, n, confidence)`
- **CI mean (unknown σ):** `ci_mean_unknown_sigma(xbar, s, n, confidence)` *(SciPy)*
- **CI proportion (Wald/Wilson):** `ci_proportion_wald(phat, n, confidence)`, `ci_proportion_wilson(phat, n, confidence)`
- **Two-proportion z test:** `two_proportion_z_test(x1_success, n1, x2_success, n2, alternative)`
- **One-sample tests:** `one_sample_z_test(...)`, `one_sample_t_test(...)` *(SciPy)*
- **Two-sample t test:** `two_sample_t_test(x1, x2, equal_var=False, alternative)` *(NumPy + SciPy)*
- **CLT simulation:** `clt_sample_means(sampler, n, reps)` *(NumPy)*


## 1. Mean with Known σ: CI and Z-Test
Use z when the population standard deviation (σ) is known.

In [ ]:
xbar, sigma, n = 102.4, 15.0, 50
conf = 0.95
low, high = ci_mean_known_sigma(xbar, sigma, n, confidence=conf)
print(f'{conf*100:.1f}% CI (known σ): ({low:.3f}, {high:.3f})')

z, p = one_sample_z_test(xbar=xbar, mu0=100.0, sigma=sigma, n=n, alternative='two-sided')
alpha = 0.05
print(f'Z = {z:.3f}, p-value = {p:.4f} -> Reject H0? {p <= alpha}')


## 2. Mean with Unknown σ: t CI and One-Sample t-Test
Uses sample standard deviation `s` and the Student's t distribution (requires SciPy).

In [ ]:
xbar, s, n = 102.4, 16.2, 50
conf = 0.95
try:
    low_t, high_t = ci_mean_unknown_sigma(xbar, s, n, confidence=conf)
    t_stat, p_t, df = one_sample_t_test(xbar=xbar, mu0=100.0, s=s, n=n, alternative='two-sided')
    print(f'{conf*100:.1f}% t-CI (unknown σ): ({low_t:.3f}, {high_t:.3f}) [df={df}]')
    print(f't = {t_stat:.3f}, df = {df}, p-value = {p_t:.4f}')
except ImportError as e:
    print('SciPy-dependent function not available:', e)


## 3. Proportions: Wald vs Wilson CI; Two-Proportion Z-Test
Wilson is recommended for small `n` or extreme `phat`. Two-proportion test uses pooled SE under H0.

In [ ]:
x_success, n_total = 18, 50
phat = x_success / n_total
wald = ci_proportion_wald(phat, n_total, confidence=0.95)
wilson = ci_proportion_wilson(phat, n_total, confidence=0.95)
print(f'phat = {phat:.3f}; Wald 95% CI: {wald}; Wilson 95% CI: {wilson}')

# Two-proportion test example: A:42/100 vs B:55/120; test A < B (i.e., p1 < p2)
z, p = two_proportion_z_test(42, 100, 55, 120, alternative='less')
print(f'Two-proportion z = {z:.3f}, p-value = {p:.4f}')


## 4. Sample Size Calculations
Compute minimum `n` to achieve a target margin of error (MoE).

In [ ]:
n_mean = sample_size_mean(moe=2.5, sigma=12, confidence=0.95)
n_prop = sample_size_proportion(moe=0.04, p_guess=0.5, confidence=0.95)
print('Required n (mean, z-based):', n_mean)
print('Required n (proportion, conservative p=0.5):', n_prop)


## 5. Central Limit Theorem (CLT) Simulation
Draw many samples from a **non-normal** population and show that their means are approximately normal.
We simulate from an exponential distribution and compare `n=10` vs `n=40`.

In [ ]:
if np is None or plt is None:
    print('NumPy/Matplotlib not available; skipping CLT demo.')
else:
    rng = np.random.default_rng(123)
    def sampler(k):
        return rng.exponential(scale=1.0, size=k)  # mean=1, var=1

    means_n10 = clt_sample_means(sampler, n=10, reps=5000)
    means_n40 = clt_sample_means(sampler, n=40, reps=5000)

    # Theoretical mean and std of sample mean for exponential(1)
    mu = 1.0
    import math
    def normal_pdf(x, mu, sigma):
        return (1.0/(math.sqrt(2*math.pi)*sigma)) * math.exp(-0.5*((x-mu)/sigma)**2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

    for ax, data, n in [(axes[0], means_n10, 10), (axes[1], means_n40, 40)]:
        ax.hist(data, bins=40, density=True, alpha=0.6, color='steelblue', edgecolor='white')
        sigma_mean = 1.0/ math.sqrt(n)
        xs = np.linspace(min(data), max(data), 400)
        ys = [normal_pdf(x, mu, sigma_mean) for x in xs]
        ax.plot(xs, ys, 'crimson', lw=2, label='N(mu, sigma/sqrt(n))')
        ax.axvline(mu, color='black', ls='--', lw=1, label='Population mean')
        ax.set_title(f'Sample means (n={n})')
        ax.set_xlabel('Sample mean')
        ax.set_ylabel('Density')
        ax.legend()

    fig.suptitle('CLT: Sample Means Approximate Normality from a Skewed Population', fontsize=14)
    plt.show()


## 6. Try It Yourself
Adjust parameters and re-run to see how conclusions change. If `ipywidgets` is installed, you'll get sliders. Otherwise, edit variables directly.

In [ ]:
# One-sample t test playground (requires SciPy)
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    USE_WIDGETS = True
except Exception:
    USE_WIDGETS = False

def run_t_demo(xbar=5.2, mu0=5.0, s=1.1, n=30, alternative='two-sided', confidence=0.95):
    print(f'Inputs: xbar={xbar}, mu0={mu0}, s={s}, n={n}, alt={alternative}, conf={confidence}')
    try:
        ci = ci_mean_unknown_sigma(xbar, s, n, confidence)
        t, p, df = one_sample_t_test(xbar, mu0, s, n, alternative)
        print(f'CI: ({ci[0]:.3f}, {ci[1]:.3f}) [df={df}]')
        print(f't = {t:.3f}, p = {p:.4f}  -> Reject H0 @ 0.05? {p <= 0.05}')
    except ImportError as e:
        print('SciPy-dependent function not available:', e)

if USE_WIDGETS:
    interact(run_t_demo,
             xbar=FloatSlider(5.2, min=3, max=8, step=0.1),
             mu0=FloatSlider(5.0, min=3, max=8, step=0.1),
             s=FloatSlider(1.1, min=0.3, max=3.0, step=0.1),
             n=IntSlider(30, min=5, max=200, step=1),
             alternative=Dropdown(options=['two-sided','less','greater'], value='two-sided'),
             confidence=FloatSlider(0.95, min=0.80, max=0.99, step=0.01));
else:
    # Fallback: run once with defaults; edit values and re-run
    run_t_demo()


## 7. Mini-Lab Tasks (For Students)
1. (Mean, known σ) A sensor reports mean temperature `xbar=71.8°F` from `n=64` readings; σ is known to be `8°F`.
   - Compute a 95% CI and interpret it in one sentence.
   - Test H0: μ=70 vs H1: μ≠70 at α=0.05.
2. (Mean, unknown σ) A sample of size `n=22` has `xbar=14.2` and `s=3.8`.
   - Compute a 95% t-CI and test H0: μ=13 vs H1: μ>13.
3. (Proportion CI) Out of `n=150` emails, `x=18` were flagged as spam.
   - Compute Wald and Wilson 95% CIs and comment on differences.
4. (Two-proportion test) Group A: `40/100`; Group B: `62/140`.
   - Test whether Group B has a higher success rate. Report z, p, and a conclusion.
5. (CLT) Using the CLT cell above, change the population to a Uniform(0, 2) distribution.
   - Re-run for `n=10` and `n=40`. What changes do you observe?


## Appendix: Cheat Sheet
**Critical values**

- `z_critical(conf)` → z*
- `t_critical(df, conf)` → t*

**CIs**

- Mean (known σ): `ci_mean_known_sigma(xbar, sigma, n, conf)`
- Mean (unknown σ): `ci_mean_unknown_sigma(xbar, s, n, conf)`
- Proportion: `ci_proportion_wald(phat, n, conf)`, `ci_proportion_wilson(phat, n, conf)`

**p-values**

- `p_value_z(z, alternative)`
- `p_value_t(t, df, alternative)`

**Tests**

- One-sample mean: `one_sample_z_test(...)`, `one_sample_t_test(...)`
- Two-sample mean: `two_sample_t_test(x1, x2, equal_var=False, alternative)`
- Two-proportion: `two_proportion_z_test(x1_success, n1, x2_success, n2, alternative)`

**CLT Simulation**

- `clt_sample_means(sampler, n, reps)`

> Normal quantiles use Acklam's approximation (with one Halley refinement).


---
**Tips for Instructors**: Duplicate this notebook for labs. Add hidden cells with solutions, or create exercise cells that import a separate `solutions.py` to reveal answers on demand.
